# Bring-up Case Study

AIM: We would like to make a new chip operational for the first time and extract fundamental properties of the system.

## Imports

In [ ]:
from laboneq._automation.utils import load_automation_parameters
from laboneq.simple import *

from laboneq_applications._automation import WorkflowAutomation, WorkflowLayer
from laboneq_applications.experiments import (  # , amplitude_rabi
    qubit_spectroscopy,
    ramsey,
)
from laboneq_applications.qpu_types.tunable_transmon import demo_platform

## Define the quantum platform and session

We define a quantum platform to emulate a 4-qubit device.

In [ ]:
# Create a demonstration QuantumPlatform for a 4-qubit tunable-transmon QPU:
qt_platform = demo_platform(n_qubits=4)

# The platform contains a setup, which is an ordinary LabOne Q DeviceSetup (1x PQSC, 1x SHFQC, 1x HDAWG):
setup = qt_platform.setup

# And a 4-qubit tunable-transmon QPU:
qpu = qt_platform.qpu

# Inside the QPU, we have quantum elements, which is a list of four LabOne Q Application
# Library TunableTransmonQubit qubits:
qubits = qpu.quantum_elements

# We connect to the session in emulation mode:
session = Session(setup)
session.connect(do_emulation=True)

## Initialize the automation framework

Typically, qubit characterization routines follow a pre-defined algorithm in the form of a _directed acyclic graph (DAG)_.

For example, assuming that the readout resonator is already characterized, the qubit characterization may involve:

1. **Qubit Spectroscopy**: sweep the drive frequency while keeping the resonant frequency fixed --> extract g-e transition frequency
2. **Rabi Experiment**: use a pulse at this g-e transition frequency and sweep the pulse amplitude --> extract drive amplitude for pi/2 pulse
3. **Ramsey Experiment**: use this amplitude to generate two pi/2 pulses separated by variable delay --> extract T2* dephasing time

<div align="center">
  <img src="images/flow_diagram.png" width="800" />
</div>

We can build this tune-up procedure using the LabOne Q Automation framework.

In [ ]:
auto = WorkflowAutomation(
    session,
    qpu=qpu,
    automation_parameters=load_automation_parameters("bring_up_case_study.yml"),
)

In [ ]:
qs1 = WorkflowLayer(
    qubit_spectroscopy.experiment_workflow,
    ["q0", "q1", "q2", "q3"],
    key="qs1",
    depends_on=["__root__"],
)

## Uncomment this layer when not in emulation mode
# rabi1 = WorkflowLayer(
#     amplitude_rabi.experiment_workflow, ["q0", "q1", "q2", "q3"], key="rabi1", depends_on=["qs1"]
# )

ramsey1 = WorkflowLayer(
    ramsey.experiment_workflow,
    ["q0", "q1", "q2", "q3"],
    key="ramsey1",
    depends_on=["qs1"],
    sequential=True,
    workflow_parameters={"q3": {"detunings": 680000.0}},
)

In [ ]:
auto.add_layer(qs1)
# auto.add_layer(rabi1)  # Uncomment this layer when not in emulation mode
auto.add_layer(ramsey1)
auto.plot();

The automation graph consists of **layers** (which may be executed in parallel or sequentially) and **nodes** (which may be executed individually).

<img src="images/automation_example.png">

### Step-by-step execution

To explain the key decision-making process in the qubit characterization, let's start by running the automation step-by-step.

#### Qubit Spectroscopy

First, we run the qubit spectroscopy for the qubits to determine their ge transition frequencies.

In [ ]:
auto.run_layer("qs1")
auto.plot();

Each experiment now includes an **evaluation task**, which defines the criteria for success and updates. For example, for the qubit spectroscopy experiment, one criterion for success is the R^2 value of a Lorentzian fit being above a certain threshold, e.g. 0.99. One criterion for updating is the change in the ge transition frequency being above a certain threshold, e.g. 5 MHz.

Example plot:

<div align="center">
    <img src="https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/images/qubit_spec_result.png">
</div>

To emulate a real set-up, we realize that qubit "q2" consistently fails and is defective. Therefore, we deactivate this node to continue with our automation routine.

In [ ]:
auto.deactivate_node("qs1_q2")
auto.plot();

In [ ]:
auto.get_layer("qs1").status

We can now examine the new _ge transition frequencies_ for each of the qubits.

In [ ]:
for i, q in enumerate(auto.qpu.quantum_elements):
    if q.uid != "q2":
        old_value = (
            auto.get_layer("qs1")
            .workflow_results[0]
            .tasks["analysis_workflow"]
            .output["old_parameter_values"][q.uid]["resonance_frequency_ge"]
        )
        new_value = (
            auto.get_layer("qs1")
            .workflow_results[0]
            .tasks["analysis_workflow"]
            .output["new_parameter_values"][q.uid]["resonance_frequency_ge"]
        )
        print(f"omega_q{i} = {old_value:.0f} -> {new_value:.0f}")

In [ ]:
auto.qpu["q0"]

#### Rabi Experiment

Having found the ge transition frequencies of our qubits, we can now apply a pulse at this frequency and sweep the pulse amplitude. The Rabi oscillations will help us to determine the drive amplitude needed for a pi/2 pulse. (NB: Since it is not possible to extract these amplitudes in emulation mode, we use dummy data for this step.)

In [ ]:
# auto.run_layer("rabi1")
# auto.plot();

Example plot:

<div align="center">
    <img src="https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/images/rabi_plot.svg">
</div>

We can now examine the new _pi/2 pulse amplitudes_ for each of the qubits.

In [ ]:
# for q_uid in ["q0", "q1", "q3"]:
#     old_value = auto.get_layer("rabi1").workflow_results[0].tasks["analysis_workflow"].output['old_parameter_values'][q_uid]['ge_drive_amplitude_pi2']
#     new_value = auto.get_layer("rabi1").workflow_results[0].tasks["analysis_workflow"].output['new_parameter_values'][q_uid]['ge_drive_amplitude_pi2']
#     print(f"A_{q_uid} = {old_value:.2f} -> {new_value.nominal_value:.2f}")

In [ ]:
# auto.qpu["q0"]

#### Ramsey Experiment

Having found the drive amplitudes needed to correctly calibrate the pi/2 pulses, we can now perform a Ramsey Interference Experiment  to determine the qubit dephasing time T2*. We do this sequentially to avoid residual ZZ coupling effects.

In [ ]:
auto.run_layer("ramsey1")
auto.plot();

We can now examine the computed _dephasing times_ T2* for each of the qubits.

In [ ]:
for i, q_uid in enumerate(["q0", "q1", "q3"]):
    ge_t2_star = (
        auto.get_layer("ramsey1")
        .workflow_results[i]
        .tasks["analysis_workflow"]
        .output["new_parameter_values"][q_uid]["ge_T2_star"]
    )
    print(f"T2*_{q_uid} = {ge_t2_star.nominal_value:.13f}")

In [ ]:
auto.qpu["q0"]

### Automated

Finally, we can demonstrate the automated bring-up and characterization of a new chip.

#### Reset the graph

In [ ]:
auto.reset()
auto.plot();

#### Run the graph

We can now run the graph using a single method. Node "qs_q2" will be run until it reaches its max fail count, after which it will be deactivated. Qubit parameters will only be updated if the node status is successful. Layer "ramsey" will be run sequentially, due to it's sequential attribute. Graph execution will only continue once the previous layer status is successful.

In [ ]:
auto.run()
auto.plot();

The chip is now ready for further characterization and tune-up!


